# Jerk Modes Demo

This notebook combines the jerk-mode comparison and the analytic
source-motion demo into one place.

It compares `fast_approx` against `accurate`, checks the high-fidelity path
against a direct-sum jerk reference, and gives a simple timing comparison.


In [ ]:
import time

import jax
import jax.numpy as jnp
import numpy as np

from jaccpot import FMMPreset, FastMultipoleMethod


In [ ]:
def sample_problem(n=256, seed=7, dtype=jnp.float32):
    key = jax.random.PRNGKey(seed)
    kpos, kmass, kvel = jax.random.split(key, 3)
    positions = jax.random.uniform(kpos, (n, 3), minval=-1.0, maxval=1.0, dtype=dtype)
    masses = jax.random.uniform(kmass, (n,), minval=0.5, maxval=1.5, dtype=dtype)
    velocities = jax.random.uniform(kvel, (n, 3), minval=-0.2, maxval=0.2, dtype=dtype)
    return positions, masses, velocities


def direct_sum_jerk(positions, masses, velocities, *, G=1.0, softening=1e-3):
    diff = positions[:, None, :] - positions[None, :, :]
    vdiff = velocities[:, None, :] - velocities[None, :, :]
    dist_sq = jnp.sum(diff * diff, axis=-1) + softening**2
    eps = jnp.finfo(positions.dtype).eps
    inv_r = jnp.where(dist_sq > 0, 1.0 / (jnp.sqrt(dist_sq) + eps), 0.0)
    inv_r3 = inv_r / dist_sq
    inv_r5 = inv_r3 / dist_sq
    eye = jnp.eye(positions.shape[0], dtype=bool)
    inv_r3 = jnp.where(eye, 0.0, inv_r3)
    inv_r5 = jnp.where(eye, 0.0, inv_r5)
    rv = jnp.sum(diff * vdiff, axis=-1)
    term = vdiff * inv_r3[..., None] - 3.0 * rv[..., None] * diff * inv_r5[..., None]
    return -G * jnp.sum(masses[None, :, None] * term, axis=1)


def rel_err(approx, reference):
    return float(jnp.linalg.norm(approx - reference) / (jnp.linalg.norm(reference) + 1e-12))


In [ ]:
positions, masses, velocities = sample_problem()
solver = FastMultipoleMethod(preset=FMMPreset.ACCURATE, basis="solidfmm")

acc_fast, jerk_fast = solver.compute_accelerations_and_jerk(
    positions,
    masses,
    velocities,
    leaf_size=16,
    max_order=4,
    theta=0.6,
    jerk_mode="fast_approx",
)

acc_acc, jerk_acc = solver.compute_accelerations_and_jerk(
    positions,
    masses,
    velocities,
    leaf_size=16,
    max_order=4,
    theta=0.6,
    jerk_mode="accurate",
)

print("acc shape:", acc_fast.shape)
print("jerk shape:", jerk_fast.shape)
print("relative ||jerk_fast - jerk_accurate|| / ||jerk_accurate||:", rel_err(jerk_fast, jerk_acc))


## Direct-Sum Spot Check

Use a smaller reference problem to compare both modes against a direct-sum jerk.


In [ ]:
n_ref = 96
pos_ref = positions[:n_ref]
mass_ref = masses[:n_ref]
vel_ref = velocities[:n_ref]

solver_ref = FastMultipoleMethod(preset=FMMPreset.ACCURATE, basis="solidfmm", theta=1e-4)
_, jerk_fast_ref = solver_ref.compute_accelerations_and_jerk(
    pos_ref,
    mass_ref,
    vel_ref,
    leaf_size=16,
    max_order=4,
    jerk_mode="fast_approx",
)
_, jerk_acc_ref = solver_ref.compute_accelerations_and_jerk(
    pos_ref,
    mass_ref,
    vel_ref,
    leaf_size=16,
    max_order=4,
    jerk_mode="accurate",
)
jerk_direct = direct_sum_jerk(pos_ref, mass_ref, vel_ref)

print("rel_err fast_approx vs direct:", rel_err(jerk_fast_ref, jerk_direct))
print("rel_err accurate    vs direct:", rel_err(jerk_acc_ref, jerk_direct))


## Timing Comparison

The `accurate` path includes the analytic source-motion contribution, so it is
worth keeping a quick timing comparison beside the accuracy comparison.


In [ ]:
def timed_call(mode, runs=3):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        _, jerk = solver.compute_accelerations_and_jerk(
            positions,
            masses,
            velocities,
            leaf_size=16,
            max_order=4,
            theta=0.6,
            jerk_mode=mode,
        )
        jax.block_until_ready(jerk)
        times.append(time.perf_counter() - t0)
    return float(np.mean(times))


t_fast = timed_call("fast_approx")
t_acc = timed_call("accurate")
print(f"fast_approx: {t_fast:.4f} s")
print(f"accurate   : {t_acc:.4f} s")
print(f"accurate/fast: {t_acc / t_fast:.3f}x")


## Notes

- `fast_approx` is usually the lower-overhead production choice.
- `accurate` includes the far-field source-motion contribution and is the
  higher-fidelity path.
- For production defaults, benchmark both modes on your own workload and
  accuracy target.
